# 11 LightGBMによる順解析比較と目標値の選定

このNotebookでは、LightGBMを順解析モデルとして評価し、逆解析に使う目標値をデータに基づいて検討します。

目標値は、モデル精度だけで自動決定しません。製品要求を優先したうえで、実測例が十分にあり、交差検証の局所誤差が小さい値を候補として示します。

In [7]:
import sys
from dataclasses import replace
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root / 'src'))

import numpy as np
import pandas as pd
from IPython.display import display

from plastech.inverse_design import (
    DEFAULT_INVERSE_TARGETS,
    build_candidate_library,
    candidate_summary,
    fit_forward_models,
    generate_interpolated_candidates,
    rank_historical_candidates,
    specifications_frame,
)
from plastech.modeling import evaluate_model_random_split
from plastech.target_selection import (
    compare_target_value,
    cross_validated_forward_predictions,
    target_value_profile,
)


## 初期の製品要求と、データ上の目標値候補を分ける

以下の値は現在の探索用仕様です。データ分析で選ぶ値は、予測しやすい領域を示すだけであり、製品要求を置き換えるものではありません。

In [8]:
base_specifications = DEFAULT_INVERSE_TARGETS
display(specifications_frame(base_specifications))

# 目標値周辺を評価する幅と、表示桁数。用途仕様に応じて変更する。
profile_settings = {
    'S03010_030010': {'rounding': 0, 'min_support': 100},  # 硬度
    'S06010_060030': {'rounding': 1, 'min_support': 100},  # 100%モジュラス
    'S13010_130010': {'rounding': 0, 'min_support': 100},  # 脆化温度
    'S01010_010010': {'rounding': 2, 'min_support': 100},  # 比重
}


,column,property,objective,target,tolerance,weight,requirement
0,S03010_030010,硬度 HDA (0s),target,78.00,3.00,1.0,78 ± 3
1,S06010_060030,100%モジュラス (JIS2 200mm/min),target,7.50,1.50,1.0,7.5 ± 1.5
2,S13010_130010,脆化温度 (JIS K6723),upper_bound,-33.00,3.00,1.0,<= -33
3,S01010_010010,比重 (23℃),target,1.34,0.05,0.7,1.34 ± 0.05


## Random ForestとLightGBMを同じ分割で比較する

LightGBMを使うこと自体は精度向上を保証しません。同じデータ・同じ分割で比較し、各物性で良いモデルを選びます。ここでの値は通常ランダム分割の比較値です。未知の材料系列への適用は、別途グループ分割でも確認してください。

In [9]:
comparison_rows = []
for spec in base_specifications:
    for model_name, n_estimators in (('random_forest', 500), ('lightgbm', 800)):
        evaluation = evaluate_model_random_split(
            model_name,
            dataset_name='processed_main',
            target_col=spec.column,
            n_estimators=n_estimators,
            random_state=42,
        )
        row = evaluation.metrics.iloc[0].to_dict()
        row['property'] = spec.name
        comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows)
display(model_comparison[['property', 'model', 'MAE', 'RMSE', 'R2', 'max_abs_error']].sort_values(['property', 'RMSE']))


,property,model,MAE,RMSE,R2,max_abs_error
3,100%モジュラス (JIS2 200mm/min),lightgbm,1.005715,1.581707,0.863359,10.935438
2,100%モジュラス (JIS2 200mm/min),random_forest,0.969587,1.592835,0.861429,11.716100
7,比重 (23℃),lightgbm,0.021375,0.038373,0.847592,0.634512
6,比重 (23℃),random_forest,0.024170,0.042356,0.814311,0.638328
0,硬度 HDA (0s),random_forest,2.940391,4.681618,0.863670,29.146100
1,硬度 HDA (0s),lightgbm,3.028799,4.735027,0.860541,27.917707
5,脆化温度 (JIS K6723),lightgbm,3.074129,4.229543,0.827522,17.202285
4,脆化温度 (JIS K6723),random_forest,3.229352,4.455805,0.808574,21.088000


## 交差検証で「選びやすい目標値の領域」を調べる

各行を学習に使わない out-of-fold（OOF）予測を作ります。候補目標値ごとに、目標値の近傍にある実測例数、近傍でのMAE、90パーセンタイル誤差を計算します。

`reliability_score` が小さいほど、必要な実測例数を満たし、かつ近傍の予測誤差が小さい値です。これは製品仕様の自動決定ではなく、逆解析の初期検証に適した目標値候補を選ぶ指標です。

In [10]:
cv_results = {}
target_profiles = {}
recommended_specifications = []

for spec in base_specifications:
    cv_result = cross_validated_forward_predictions(
        spec.column,
        dataset_name='processed_main',
        model_name='lightgbm',
        n_splits=5,
        n_estimators=800,
        random_state=42,
    )
    settings = profile_settings[spec.column]
    value_grid = list(np.linspace(cv_result.predictions['actual'].quantile(0.1), cv_result.predictions['actual'].quantile(0.9), 17))
    value_grid.append(spec.target)  # 現在の目標値も必ず比較する
    profile = target_value_profile(
        cv_result,
        target_tolerance=spec.tolerance,
        target_values=value_grid,
        rounding=settings['rounding'],
        min_support=settings['min_support'],
    )
    cv_results[spec.column] = cv_result
    target_profiles[spec.column] = compare_target_value(profile, spec.target)
    best = profile[profile['eligible']].iloc[0]
    recommended_specifications.append(replace(spec, target=float(best['target_value'])))

cv_metrics = pd.concat([result.metrics for result in cv_results.values()], ignore_index=True)
display(cv_metrics)


,model,target_col,rows,n_splits,MAE,RMSE,R2,p90_abs_error
0,lightgbm,S03010_030010,3046,5,3.353992,5.226010,0.833656,7.798232
1,lightgbm,S06010_060030,3838,5,1.062724,1.731907,0.842061,2.534269
2,lightgbm,S13010_130010,3392,5,3.114887,4.337227,0.816022,6.721997
3,lightgbm,S01010_010010,5088,5,0.021732,0.037077,0.855412,0.047061


In [11]:
for spec in base_specifications:
    print(f'\n{spec.name}: 現在の目標値 = {spec.description()}')
    display(target_profiles[spec.column].head(8))

print('\nデータ密度とOOF誤差に基づく探索用の候補値')
display(specifications_frame(tuple(recommended_specifications)))



硬度 HDA (0s): 現在の目標値 = 78 ± 3


,target_value,window,support,support_rate,local_MAE,local_RMSE,local_p90_abs_error,local_bias,eligible,reliability_score,is_current_target
0,89.0,3.0,615,0.201904,2.614427,3.904626,5.962069,-1.134030,True,3.966888,False
1,91.0,3.0,559,0.183519,2.602075,3.986125,6.104710,-1.533679,True,4.020662,False
2,87.0,3.0,641,0.210440,2.750861,4.033447,6.228102,-1.116741,True,4.075608,False
3,78.0,3.0,507,0.166448,2.710508,3.778281,5.957160,-0.295967,True,4.200068,True
4,85.0,3.0,583,0.191399,2.850916,4.189296,6.426999,-1.036981,True,4.239997,False
5,72.0,3.0,590,0.193697,2.870745,4.366636,6.329274,0.572656,True,4.251562,False
6,74.0,3.0,520,0.170716,2.782624,4.081412,6.002971,0.008053,True,4.253447,False
7,76.0,3.0,458,0.150361,2.720669,3.837032,5.716748,-0.237233,True,4.287887,False



100%モジュラス (JIS2 200mm/min): 現在の目標値 = 7.5 ± 1.5


,target_value,window,support,support_rate,local_MAE,local_RMSE,local_p90_abs_error,local_bias,eligible,reliability_score,is_current_target
0,3.7,1.5,1028,0.267848,0.821643,1.508461,2.013187,0.579227,True,1.153098,False
1,5.7,1.5,1299,0.338458,0.861640,1.463154,2.011661,0.424080,True,1.156500,False
2,4.4,1.5,1185,0.308755,0.850841,1.555192,2.017855,0.543316,True,1.159559,False
3,5.0,1.5,1280,0.333507,0.867776,1.556397,2.105785,0.530520,True,1.164817,False
4,6.4,1.5,1211,0.315529,0.883958,1.456290,2.037535,0.370251,True,1.189344,False
5,7.0,1.5,1091,0.284263,0.883278,1.422066,1.976582,0.310758,True,1.205020,False
6,7.5,1.5,1032,0.268890,0.887758,1.366717,1.975301,0.249441,True,1.218570,True
7,7.7,1.5,1001,0.260813,0.902234,1.361262,2.065533,0.229163,True,1.238129,False



脆化温度 (JIS K6723): 現在の目標値 = <= -33


,target_value,window,support,support_rate,local_MAE,local_RMSE,local_p90_abs_error,local_bias,eligible,reliability_score,is_current_target
0,-41.0,3.0,764,0.225236,2.506604,3.521283,5.654725,1.083954,True,3.633531,False
1,-40.0,3.0,762,0.224646,2.563546,3.550640,5.899367,1.032922,True,3.691950,False
2,-44.0,3.0,630,0.185731,2.505593,3.635109,5.421551,1.382582,True,3.746593,False
3,-43.0,3.0,702,0.206958,2.582866,3.609630,5.901066,1.320965,True,3.758505,False
4,-38.0,3.0,766,0.225825,2.813478,3.768596,6.167941,1.014680,True,3.938933,False
5,-28.0,3.0,762,0.224646,2.822722,3.765002,5.733905,-0.209028,True,3.951126,False
6,-30.0,3.0,814,0.239976,2.884322,3.731797,5.815058,0.122046,True,3.976090,False
7,-31.0,3.0,804,0.237028,2.947396,3.808261,6.079245,0.289731,True,4.045932,False



比重 (23℃): 現在の目標値 = 1.34 ± 0.05


,target_value,window,support,support_rate,local_MAE,local_RMSE,local_p90_abs_error,local_bias,eligible,reliability_score,is_current_target
0,1.36,0.05,1968,0.386792,0.017888,0.025288,0.038964,0.000997,True,0.022787,False
1,1.34,0.05,1943,0.381879,0.018453,0.026284,0.041020,0.002938,True,0.023383,True
2,1.33,0.05,1938,0.380896,0.018532,0.026409,0.041018,0.003753,True,0.023469,False
3,1.30,0.05,1703,0.334709,0.018332,0.026714,0.040318,0.005733,True,0.023598,False
4,1.31,0.05,1789,0.351612,0.018476,0.026768,0.040812,0.005235,True,0.023614,False
5,1.38,0.05,1872,0.367925,0.018743,0.026888,0.042339,-0.001061,True,0.023766,False
6,1.39,0.05,1814,0.356525,0.019050,0.027495,0.043000,-0.002675,True,0.024153,False
7,1.28,0.05,1532,0.301101,0.018987,0.027888,0.042424,0.006436,True,0.024539,False



データ密度とOOF誤差に基づく探索用の候補値


,column,property,objective,target,tolerance,weight,requirement
0,S03010_030010,硬度 HDA (0s),target,89.00,3.00,1.0,89 ± 3
1,S06010_060030,100%モジュラス (JIS2 200mm/min),target,3.70,1.50,1.0,3.7 ± 1.5
2,S13010_130010,脆化温度 (JIS K6723),upper_bound,-41.00,3.00,1.0,<= -41
3,S01010_010010,比重 (23℃),target,1.36,0.05,0.7,1.36 ± 0.05


## LightGBMで逆解析候補を作る

`USE_DATA_SUPPORTED_TARGETS` を `False` にすると製品要求として設定した初期値を使います。`True` にすると、上のOOF結果で選ばれた「データが多く局所誤差が小さい」探索用候補値を使います。教授への報告では、両者を分けて記載してください。

In [12]:
USE_DATA_SUPPORTED_TARGETS = False
search_specifications = (
    tuple(recommended_specifications) if USE_DATA_SUPPORTED_TARGETS else base_specifications
)

lightgbm_models = fit_forward_models(
    search_specifications,
    dataset_name='processed_main',
    model_name='lightgbm',
    n_estimators=800,
    random_state=42,
)
library = build_candidate_library(lightgbm_models, dataset_name='processed_main')
historical_ranking = rank_historical_candidates(library, lightgbm_models)
virtual_candidates = generate_interpolated_candidates(
    library,
    historical_ranking,
    lightgbm_models,
    seed_count=20,
    partners_per_seed=3,
    fractions=(0.25, 0.5, 0.75),
    random_state=42,
)

print('採用した目標値')
display(specifications_frame(search_specifications))
print(f'実測値で全目標を満たす既存配合: {historical_ranking["observed_meets_all"].sum():,} 件')
print(f'予測上で全目標を満たす仮想候補: {virtual_candidates["predicted_meets_all"].sum():,} 件')
display(candidate_summary(historical_ranking, lightgbm_models, top_n=10))
display(candidate_summary(virtual_candidates, lightgbm_models, top_n=10))


採用した目標値


,column,property,objective,target,tolerance,weight,requirement
0,S03010_030010,硬度 HDA (0s),target,78.00,3.00,1.0,78 ± 3
1,S06010_060030,100%モジュラス (JIS2 200mm/min),target,7.50,1.50,1.0,7.5 ± 1.5
2,S13010_130010,脆化温度 (JIS K6723),upper_bound,-33.00,3.00,1.0,<= -33
3,S01010_010010,比重 (23℃),target,1.34,0.05,0.7,1.34 ± 0.05


実測値で全目標を満たす既存配合: 80 件
予測上で全目標を満たす仮想候補: 38 件


,candidate_type,row_id,primary_pvc,primary_plasticizer,observed_score,observed_meets_all,predicted_score,predicted_meets_all,observed__S03010_030010,predicted__S03010_030010,observed__S06010_060030,predicted__S06010_060030,observed__S13010_130010,predicted__S13010_130010,observed__S01010_010010,predicted__S01010_010010
0,historical,CP0000000002417,PVC1000 / TH1000(GW) (PVC.15),フタル酸エステル系 / DOP (可塑剤オイル類.29),0.068108,True,0.305404,True,78.0,79.263276,7.5,7.174387,-35.0,-33.529120,1.322,1.304870
1,historical,CP0000000000502,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.131712,True,0.137728,True,78.0,78.070786,7.0,6.781479,-40.0,-38.583881,1.329,1.339501
2,historical,CP1000000009335,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.131712,True,0.137728,True,78.0,78.070786,7.0,6.781479,-40.0,-38.583881,1.329,1.339501
3,historical,CP0000000000958,PVC1300 / TH1300(JG) (PVC.23),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.205586,True,0.391725,False,77.0,76.518896,7.3,7.255884,-36.0,-31.674637,1.319,1.314918
4,historical,CP0000000002767,PVC2251- / TH2500(SG) (PVC.32),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.207568,True,0.308563,True,78.0,78.388883,6.6,6.114235,-38.0,-40.119922,1.328,1.333699
5,historical,CP0000000000041,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.216757,True,0.414452,True,78.0,80.457935,7.2,7.605976,-34.0,-34.851973,1.383,1.385965
6,historical,CP1000000008263,PVC1000 / TH1000(GW) (PVC.15),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.224505,True,0.247161,True,77.0,77.388499,7.3,6.764926,-36.0,-39.290765,1.314,1.324242
7,historical,CP0000000002534,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DOP (可塑剤オイル類.29),0.227568,True,0.361016,True,78.0,77.194203,6.3,6.114903,-33.0,-37.445220,1.343,1.350269
8,historical,CP0000000002690,PVC2000 / TH2000(SW) (PVC.30),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.228468,True,0.150095,True,79.0,78.279333,6.9,6.814857,-37.0,-37.227907,1.332,1.339609
9,historical,CP0000000000501,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.236036,True,0.240487,True,78.0,77.028010,6.4,6.849272,-39.0,-37.305940,1.350,1.349428


,candidate_type,row_id,parent_a,parent_b,parent_a_fraction,primary_pvc,primary_plasticizer,predicted_score,predicted_meets_all,predicted__S03010_030010,predicted__S06010_060030,predicted__S13010_130010,predicted__S01010_010010
0,interpolated_virtual,virtual::CP1000000009335::CP1000000009866::0.75,CP1000000009335,CP1000000009866,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.079374,True,78.122053,7.300811,-40.384941,1.331414
1,interpolated_virtual,virtual::CP1000000005638::CP1000000008187::0.75,CP1000000005638,CP1000000008187,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.097912,True,78.432629,7.532975,-37.855714,1.354006
2,interpolated_virtual,virtual::CP0000000000501::CP1000000009781::0.75,CP0000000000501,CP1000000009781,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.164216,True,78.334959,7.485191,-40.130973,1.305281
3,interpolated_virtual,virtual::CP0000000000502::CP0000000000501::0.75,CP0000000000502,CP0000000000501,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.178377,True,78.282431,6.768803,-37.656069,1.345599
4,interpolated_virtual,virtual::CP0000000000502::CP0000000000501::0.50,CP0000000000502,CP0000000000501,0.50,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.182920,True,78.211696,6.739166,-37.574806,1.347072
5,interpolated_virtual,virtual::CP0000000002767::CP1000000000648::0.75,CP0000000002767,CP1000000000648,0.75,PVC2251- / TH2500(SG) (PVC.32),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.188547,True,77.730351,6.852111,-38.850118,1.327442
6,interpolated_virtual,virtual::CP1000000009335::CP1000000007634::0.75,CP1000000009335,CP1000000007634,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.202785,True,79.096511,6.932186,-38.677060,1.339553
7,interpolated_virtual,virtual::CP1000000008238::CP1000000008239::0.75,CP1000000008238,CP1000000008239,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.206840,True,78.104945,6.668097,-41.403157,1.327448
8,interpolated_virtual,virtual::CP0000000000502::CP0000000000501::0.25,CP0000000000502,CP0000000000501,0.25,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.207139,True,77.571608,6.792208,-37.412147,1.350840
9,interpolated_virtual,virtual::CP0000000000502::CP1000000007413::0.75,CP0000000000502,CP1000000007413,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.220116,True,79.415501,7.899079,-36.805543,1.345467


## 解釈上の注意

- LightGBMのR²やMAEがRandom Forestより良い物性だけを、LightGBM採用候補とする。
- OOF局所MAEが小さい目標値は、探索実験を始めやすい領域であって、用途上望ましい物性を意味しない。
- 仮想候補はモデル予測であり、処方可否・加工性・コスト・法規を保証しない。
- 教授への報告では、ランダム分割だけでなく、主PVC・主可塑剤を分けたグループ評価も追加して未知系列への精度を確認する。